In [9]:
##ai agent class assignment
##Movie Expert Agent

import openai, json

client = openai.OpenAI()

messages = []


user_profile = {
    "favorite_genres": [],
    "watched_movies": []
}

In [10]:
import requests

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


##/movies에서 인기 영화를 가져옵니다.
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)

##/movies/:id에서 영화 정보를 가져옵니다.
def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)

##/movies/:id/credits에서 출연진 및 제작진을 가져옵니다.
def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)

##좋아하는 장르 추가합니다.
def add_favorite_genre(genre):
    if genre not in user_profile["favorite_genres"]:
        user_profile["favorite_genres"].append(genre)

    return json.dumps({
        "status": "success",
        "favorite_genres": user_profile["favorite_genres"]
    })

##영화 시청 기록 추가합니다.
def add_watched_movie(movie_title):
    if movie_title not in user_profile["watched_movies"]:
        user_profile["watched_movies"].append(movie_title)

    return json.dumps({
        "status": "success",
        "watched_movies": user_profile["watched_movies"]
    })

##사용자 프로필 조회합니다.
def get_user_profile():
    return json.dumps(user_profile, ensure_ascii=False)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "add_favorite_genre": add_favorite_genre,
    "add_watched_movie": add_watched_movie,
    "get_user_profile": get_user_profile,
}

In [11]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": """
            Get the list of currently popular movies.

            Use this function when the user asks:
            - What movies are popular now?
            - Popular movies
            - Trending movies
            - 현재 인기 영화
            - 인기 영화 추천
            """,
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": """
            Get detailed information about a movie by its movie ID.

            Use this function when the user asks:
            - What movie is ID 550?
            - Tell me about movie 550
            - 영화 550 정보 알려줘
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": """
            Get cast and crew information for a movie.

            Use this function when the user asks:
            - Who starred in movie 550?
            - Cast of movie 550
            - 출연진 알려줘
            - 감독 알려줘
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_favorite_genre",
            "description": """
            Save user's favorite movie genre.

            Example:
            - 나는 액션 영화를 좋아해
            - SF 장르 좋아함
            - 코미디 영화 좋아해
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "genre": {
                        "type": "string"
                    }
                },
                "required": ["genre"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_watched_movie",
            "description": """
            Save a movie already watched by the user.

            Example:
            - 인터스텔라 봤어
            - Fight Club 이미 봤어
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_title": {
                        "type": "string"
                    }
                },
                "required": ["movie_title"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_profile",
            "description": "Get saved user preferences",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
    
]

In [12]:
SYSTEM_PROMPT = """
You are a Personal Movie Recommendation Agent.

You have access to the following tools.

Movie Tools

1. get_popular_movies()
   - Get currently popular movies.

2. get_movie_details(id)
   - Get detailed information about a movie.

3. get_movie_credits(id)
   - Get cast and crew information.

Memory Tools

4. add_favorite_genre(genre)
   - Save a user's favorite genre.

5. add_watched_movie(movie_title)
   - Save a movie the user has already watched.

6. get_user_profile()
   - Retrieve the user's saved preferences.

Rules:

1. When a user says they like a genre,
   use add_favorite_genre().

2. When a user says they watched a movie,
   use add_watched_movie().

3. Do NOT recommend movies automatically.

4. Recommend movies ONLY when the user explicitly asks for recommendations.

5. Before recommending movies,
   use get_user_profile().

6. Never recommend movies already watched.

7. Prefer movies matching favorite genres.

8. If movie information is requested,
   use the movie tools.

9. Use conversation history to personalize recommendations.

10. Always answer in Korean.

11. Always answer in plain text.

12. Do not use markdown

13. Do not use:
*, -, #, bullet points, numbered lists.

14. When saving user preferences,
    do not explain that data was saved.

15. Do not say:
    - 저장했습니다
    - 기억했습니다
    - 이 정보는 저장되었습니다

16. Simply continue the conversation naturally.
"""

In [13]:
messages.append({
    "role": "system",
    "content": SYSTEM_PROMPT
})

In [7]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    } 
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            # print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            
            result = function_to_run(**arguments)

           # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result,
            })

        call_ai()

    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


In [14]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()

User: 나는 sf 영화를 좋아해
AI: SF 영화를 좋아하시는군요! 다른 장르에 대해서도 말씀해 주시면 좋겠어요.
User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 인셉션과 인터스텔라를 이미 보셨군요. 다른 영화에 대해 더 이야기해 주시거나, 추천을 원하시면 말씀해 주세요!
User: 오늘 밤에 뭐 볼지 추천해 줄래?
AI: 추천할 만한 영화로 "Project Hail Mary"를 추천합니다. 

영화의 줄거리는 과학 선생님인 라이랜드 그레이스가 자신의 정체성과 우주에서의 임무를 찾기 위해 고군분투하는 내용입니다. 이 영화는 SF 장르에 맞아 당신이 좋아할 것으로 생각합니다.

혹시 다른 영화에도 관심이 있으시면 말씀해 주세요!
User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
AI: 당신이 좋아하는 장르는 SF이고, 이미 본 영화는 "인셉션"과 "인터스텔라"입니다. 더 궁금한 점이 있거나 다른 내용을 필요로 하시면 말씀해 주세요!
